<a href="https://colab.research.google.com/github/Magwict/Hands-on-Large-Language-Model-Code-/blob/main/Chapter_5_%E6%96%87%E6%9C%AC%E8%81%9A%E7%B1%BB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install bertopic datasets openai datamapplot

In [ ]:
!pip install datasets --upgrade

In [ ]:
#从hugging face下载数据集
from datasets import load_dataset
dataset = load_dataset("maartengr/arxiv_nlp")["train"]

#提取元数据
abstracts = dataset["Abstracts"]
titles = dataset["Titles"]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/617 [00:00<?, ?B/s]

data.csv:   0%|          | 0.00/53.2M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
from sentence_transformers import SentenceTransformer

#为每一个摘要创造嵌入
embedding_model = SentenceTransformer("thenlper/gte-small")
embeddings = embedding_model.encode(abstracts,show_progress_bar=True)

Batches:   0%|          | 0/1405 [00:00<?, ?it/s]

In [ ]:
from umap import UMAP

#将输入嵌入从384维降低到5维
umap_model = UMAP(
    n_components=5,min_dist=0.0,metric='cosine',random_state=42
)

reduced_embeddings = umap_model.fit_transform(embeddings)

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
from hdbscan import HDBSCAN

#拟合模型并提取聚类
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    metric='euclidean',
    cluster_selection_method='eom'
).fit(reduced_embeddings)

clusters = hdbscan_model.labels_ #返回每个样本的簇标签，格式为整数数组

#打印聚类结果，簇的数量
len(set(clusters))

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


155

In [ ]:
from bertopic import BERTopic

#用之前定义的模型训练新模型：使用 BERTopic 库训练一个主题建模模型
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model, #降维模型，可以用PCA替换UMAP
    hdbscan_model=hdbscan_model, #聚类模型，可以用k-means替换hdbscan
    verbose=True
).fit(abstracts,embeddings) #输入摘要和高维向量

2025-05-28 13:37:55,053 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-05-28 13:38:52,268 - BERTopic - Dimensionality - Completed ✓
2025-05-28 13:38:52,270 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-05-28 13:38:54,284 - BERTopic - Cluster - Completed ✓
2025-05-28 13:38:54,305 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-05-28 13:38:58,110 - BERTopic - Representation - Completed ✓


现在，让我们探寻由以上代码生成的主题建模。

In [ ]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,14210,-1_of_the_and_to,"[of, the, and, to, in, we, language, for, that...",[ GPT-$3$ has attracted lots of attention due...
1,0,2316,0_speech_asr_recognition_end,"[speech, asr, recognition, end, acoustic, spea...",[ The speech chain mechanism integrates autom...
2,1,2183,1_question_qa_questions_answer,"[question, qa, questions, answer, answering, a...",[ Text-based Question Generation (QG) aims at...
3,2,941,2_translation_nmt_machine_bleu,"[translation, nmt, machine, bleu, neural, engl...","[ Recently, the development of neural machine..."
4,3,880,3_summarization_summaries_summary_abstractive,"[summarization, summaries, summary, abstractiv...",[ Sentence summarization shortens given texts...
...,...,...,...,...,...
150,149,54,149_sentence_embeddings_sts_embedding,"[sentence, embeddings, sts, embedding, similar...",[ Sentence representation at the semantic lev...
151,150,54,150_gans_gan_adversarial_generation,"[gans, gan, adversarial, generation, generativ...",[ Text generation is of particular interest i...
152,151,54,151_coherence_discourse_paragraph_text,"[coherence, discourse, paragraph, text, cohesi...",[ While there has been significant progress t...
153,152,53,152_chatgpt_its_openai_tasks,"[chatgpt, its, openai, tasks, has, ai, capabil...","[ Over the last few years, large language mod..."


我们可以根据这个表格查看每个簇并打印出相关的关键词。

In [ ]:
#打印与第11个簇相关的关键词
topic_model.get_topic(11)

[('parsing', np.float64(0.04075430955451941)),
 ('dependency', np.float64(0.0335918911692885)),
 ('parser', np.float64(0.026405190580623305)),
 ('parsers', np.float64(0.01906459384296242)),
 ('treebank', np.float64(0.015824337720041225)),
 ('trees', np.float64(0.015126010102075307)),
 ('transition', np.float64(0.014436295818998192)),
 ('treebanks', np.float64(0.01277616831171186)),
 ('constituency', np.float64(0.012466041865879487)),
 ('syntactic', np.float64(0.011417735960230217))]

我们可以用find_topics()函数,用特定的搜索词去查找指定的topics。

In [ ]:
topic_model.find_topics("topic modeling")

([24, -1, 38, 32, 84],
 [np.float32(0.9545274),
  np.float32(0.9123676),
  np.float32(0.9080541),
  np.float32(0.9053283),
  np.float32(0.90453553)])

-1是所有噪声（离散值）的集合，我们不给予关注。簇24有着与我们的搜索词"topic modeling"最高的相似度，高达0.954，接下来我们查看簇24。

In [ ]:
topic_model.get_topic(24)

[('topic', np.float64(0.06811153301756999)),
 ('topics', np.float64(0.035746717561104396)),
 ('lda', np.float64(0.016020062969070364)),
 ('latent', np.float64(0.013574936227317968)),
 ('documents', np.float64(0.013201698266173009)),
 ('document', np.float64(0.012912590658853182)),
 ('modeling', np.float64(0.012084716289729468)),
 ('dirichlet', np.float64(0.01010281253111858)),
 ('word', np.float64(0.008653858081603273)),
 ('allocation', np.float64(0.007950503995528465))]

如果我们知道某个特定主题的论文标题，我们可以查找它是否在某个特定主题的簇中。

In [ ]:
topic_model.topics_[titles.index('BERTopic: Neural topic modeling with a class-based TF-IDF procedure')]

24

In [ ]:
topic_model.topics_[titles.index('Attention Is All You Need')]

2

In [ ]:
reduced_embeddings = UMAP(
    n_components=2, #指定维数2或3
    min_dist=0.0, #控制点之间的最小间距，0表示允许点重叠
    metric='cosine',
    random_state=42 #固定随机种子
).fit_transform(embeddings) #二维数组，size为（44949,2）

#将主题和文档可视化
#visualize_documents创建交互式2D散点图，每个点代表一个文档，
#点的颜色表示文档所属主题，鼠标悬停显示文档标题（titles）
fig = topic_model.visualize_documents(
    titles,
    reduced_embeddings=reduced_embeddings,
    width=1200,
    hide_annotations=True
)

#更新图例字体，更容易可视化
fig.update_layout(font=dict(size=16))

In [ ]:
#可视化条形图与排名关键字
topic_model.visualize_barchart(top_n_topics=10)

#主题之间的可视化关系
topic_model.visualize_heatmap(n_clusters=30)

#可视化主题的潜在层次结构
topic_model.visualize_hierarchy()